# Forward & Backward Pass on a Small Network

Trace **activations** in the forward pass and **gradients** stored on parameters after `loss.backward()`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
torch.manual_seed(42)
np.random.seed(42)

class DummyDataGenerator:
    """Synthetic classification data for CPU-only tutorials."""
    def __init__(self, n_samples=256, n_features=8, n_classes=3, seed=42):
        g = torch.Generator().manual_seed(seed)
        self.X = torch.randn(n_samples, n_features, generator=g)
        self.y = torch.randint(0, n_classes, (n_samples,), generator=g)

    def tensors(self):
        return self.X, self.y

class SimpleMLP(nn.Module):
    def __init__(self, in_dim=8, hidden=16, n_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, n_classes),
        )
    def forward(self, x):
        return self.net(x)


## 1. Forward pass — record layer outputs

In [ ]:
gen = DummyDataGenerator(n_samples=4, n_features=8, n_classes=3)
X, y = gen.tensors()
model = SimpleMLP(in_dim=8, hidden=16, n_classes=3)

h = model.net[0](X)
logits = model.net[2](F.relu(h))
loss = F.cross_entropy(logits, y)
print(f"logits shape: {logits.shape}, loss: {loss.item():.4f}")

## 2. Visualize forward activations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(h.detach().numpy(), aspect='auto', cmap='viridis')
axes[0].set_title('Hidden activations (batch × hidden)'); axes[0].set_xlabel('neuron')
axes[1].imshow(logits.detach().numpy(), aspect='auto', cmap='coolwarm')
axes[1].set_title('Output logits'); axes[1].set_xlabel('class')
plt.tight_layout(); plt.show()

## 3. Backward pass — gradients on weights

In [ ]:
model.zero_grad()
loss.backward()
w1_grad = model.net[0].weight.grad
w2_grad = model.net[2].weight.grad
print(f"W1 grad norm: {w1_grad.norm():.4f}, W2 grad norm: {w2_grad.norm():.4f}")

## 4. Plot gradient magnitudes per layer

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(w1_grad.numpy(), aspect='auto', cmap='RdBu_r')
axes[0].set_title('∂L/∂W1 (input→hidden)')
axes[1].imshow(w2_grad.numpy(), aspect='auto', cmap='RdBu_r')
axes[1].set_title('∂L/∂W2 (hidden→output)')
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
layers = ['W1', 'b1', 'W2', 'b2']
norms = [model.net[i].weight.grad.norm().item() if hasattr(model.net[i], 'weight')
         else model.net[i].bias.grad.norm().item() for i in [0, 0, 2, 2]]
# fix layer names
params = [('W1', model.net[0].weight), ('b1', model.net[0].bias),
          ('W2', model.net[2].weight), ('b2', model.net[2].bias)]
names, gnorms = zip(*[(n, p.grad.norm().item()) for n, p in params])
ax.bar(names, gnorms, color='teal', edgecolor='black')
ax.set_title('Gradient L2 norm per parameter tensor')
ax.set_ylabel('||grad||')
plt.tight_layout(); plt.show()